# Portfolio - Part 2
## All to be checked

### Ex. 1

In [ ]:
import numpy as np
import pandas as pd

#Make 8x7 array of random numbers
data = np.random.rand(8,7)

#Make dataframe, assinging row and column labels
df = pd.DataFrame(data, columns = ["Rand# 1", "Rand# 2", "Rand# 3", "Rand# 4", "Rand# 5", "Rand# 6", "Rand# 7"], index = ["A","B","C","D","E","F","G","H"])

print(df.info())    #Display summary
print('\n\nShape: ', df.shape)  #Display shape
print('\n\n', df.describe())    #Display statistics

### Ex. 2

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

#Load data and remove non-numerical columns
data = sns.load_dataset("iris")
num_data = data.select_dtypes(include='number')

#Create figure and boxplot chart, refine the details and render
plt.figure(figsize=(10, 6))
plt.boxplot(numeric_data.values, tick_labels=numeric_data.columns)
plt.title('Boxplot of each Iris Dataset Feature')
plt.ylabel('cm')
plt.grid(True)
plt.tight_layout()
plt.show()

### Ex. 3

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

#Load data and apply into dataset
data = sns.load_dataset("tips")
df = pd.DataFrame(data)

#Display data in sunburst chart
fig = px.sunburst(df, path=['sex', 'day', 'time'], values='total_bill')
fig.show()

### Ex. 4 - WIP

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

#Define function to calculate root mean square error
def rmse(targets, predictions):
    return np.sqrt(np.mean(np.square(predictions - targets)))

#Load data
data = sns.load_dataset("tips")
df = pd.DataFrame(data)

#Define inputs and targets
inputs, targets = df[["total_bill"]], df['tip']

#Set up model, run prediction and calculate loss
model = LinearRegression().fit(inputs, targets)
guesses = model.predict(inputs)
loss = rmse(targets, guesses)
print('Loss:', loss)

#Split train and test from dataset
inputs_train, inputs_test, targets_train, targets_test = train_test_split(inputs, targets, test_size=0.1)

#Run each set through model
model = LinearRegression().fit(inputs_train, targets_train)
predictions_train = model.predict(inputs_train)

model = LinearRegression().fit(inputs_test, targets_test)
predictions_test = model.predict(inputs_test)

#Using model, predict tip for given total bill
tb = 200
quip_tip = {"total_bill": [tb]}

tp = pd.DataFrame(quip_tip)
tip_prediction = model.predict(tp)
print("The predicted tip for a bill of ",tb, "is", round(tip_prediction[0],2))



### Ex. 5 - WIP {Tut8.1}

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.tree import plot_tree, export_text
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import itertools

#Load dataset
dataset_path = 'https://raw.githubusercontent.com/Koldim2001/test_api/refs/heads/main/titanic.csv' 
df = pd.read_csv(dataset_path)

#Take out unwanted columns and clean data
df = df[['Survived','Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]  # The subset (columns) we selected for this project
df = df.dropna(subset=['Age'])
df.drop(columns='Survived')

#Assign hotcodes to categorical data
sex_codes = {'female': 0, 'male': 1}
df['Sex'] = df.Sex.map(sex_codes)
embark_codes = {'S': 0, 'Q': 1, 'C':2}
df['Embarked'] = df.Embarked.map(embark_codes)

#Split test and train datasets off
train, test = train_test_split(df, test_size=0.2)

#Define a function that plots a confusion matrix
def plot_confusion_matrix(cm, classes, normalize=False, title='Confusion matrix', cmap=plt.cm.Blues):
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    #Render and display confusion matrix
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    #Graph annotations
    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

#Define a function that runs the model experiment
def experiment(max_depth, min_samples_split):

    #Build and train Decision Tree model
    model = DecisionTreeClassifier(max_depth=max_depth, min_samples_split=min_samples_split, random_state=42)
    model.fit(train.drop('Survived', axis=1), train['Survived'])

    #Calculate accuracy metrics
    preds = model.predict(test.drop('Survived', axis=1))
    acc = accuracy_score(test['Survived'], preds)
    cm = confusion_matrix(test['Survived'], preds)

    print("Accuracy: ", round(acc*100,2), "%")

    #Plot confusion matrix and classification report
    plot_confusion_matrix(cm, classes=['Not Survived', 'Survived'])
    report = classification_report(test['Survived'], preds, target_names=['Not Survived', 'Survived'])
    print(report)

    #Save model
    with open('../outputs/models/p2model_dt.pkl', 'wb') as f:
        pickle.dump(model, f)
        
max_depth = 5
min_samples_split = 150

experiment(max_depth, min_samples_split)



In [ ]:
#Open model
with open('../outputs/models/p2model_dt.pkl', 'rb') as f:
    model = pickle.load(f)


#Predict outcome of Titanic trip for a person
person = pd.DataFrame({'Pclass':[1],'Sex':[0],'Age':[55],'SibSp':[0],'Parch':[2],'Fare':[7.2500],'Embarked':[1]})
prediction = model.predict(person)
print(f"The model predicts {prediction}")

#Translate prediction
if prediction == [1]:
    print ("This person is, the most likely, is a survivor.")
else:
    print("This person, the most likely, perished.")

#Plot feature importance from Decision Tree
importance_df = pd.DataFrame({
    'Feature': df.drop(columns='Survived').columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)
plt.title('Feature Importance')
sns.barplot(data=importance_df.head(10), x='Importance', y='Feature', hue='Importance')